# Prepare shared CXR metadata

This notebook creates the shared study-level train/validation/test CSV files used by the repository utilities. The maintained Qwen workflow in `09_qwen3vl_cxr.ipynb` remains the full data-preparation and model notebook. This notebook is a lightweight alternative when you only need the common table.

**Data source:** Indiana University chest X-rays through KaggleHub. Keep credentials in Kaggle/Colab secrets; never paste tokens into notebook cells. Images remain in the KaggleHub cache; the CSV stores image filenames. Splits are assigned by study UID to prevent views from a study crossing splits.

In [ ]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Drive mount skipped (not running in Colab).")

# The checked-out working folder is primary; GitHub is only a fresh-Colab fallback.
CANDIDATES = [Path.cwd(), Path("/content/cxr_second_reader"), Path("/content/drive/MyDrive/cxr_second_reader")]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "config.py").exists()), None)
if REPO_ROOT is None:
    if not Path("/content").exists():
        raise FileNotFoundError("Open this notebook from the cxr_second_reader project folder.")
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/eyadstein/cxr_second_reader.git", "/content/cxr_second_reader"], check=True)
    REPO_ROOT = Path("/content/cxr_second_reader")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Project folder:", REPO_ROOT)


In [ ]:
%pip -q install pandas numpy scikit-learn pillow kagglehub rouge-score nltk


In [ ]:
import re
import numpy as np
import pandas as pd
import kagglehub
from src.config import SHARED, INDIANA_SLUG

SHARED = Path(SHARED)
SHARED.mkdir(parents=True, exist_ok=True)
source = Path(kagglehub.dataset_download(INDIANA_SLUG))
report_files = list(source.rglob("indiana_reports.csv"))
projection_files = list(source.rglob("indiana_projections.csv"))
if not report_files or not projection_files:
    raise FileNotFoundError(f"Indiana report/projection CSVs were not found under {source}")
reports = pd.read_csv(report_files[0]).fillna("")
projections = pd.read_csv(projection_files[0]).fillna("")
print("Studies:", len(reports), "projection rows:", len(projections))


In [ ]:
def clean_text(value):
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"\bX{3,}\b", " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

for column in ("findings", "impression", "indication"):
    if column not in reports:
        reports[column] = ""
    reports[column] = reports[column].map(clean_text)

reports["target_text"] = reports.apply(
    lambda row: " ".join(
        part for part in (
            f"Findings: {row.findings}" if row.findings else "",
            f"Impression: {row.impression}" if row.impression else "",
        ) if part
    ), axis=1
)
reports = reports[reports.target_text.str.strip().ne("")].drop_duplicates("uid").copy()
frontal = projections[projections.projection.astype(str).str.lower().eq("frontal")].drop_duplicates("uid")[["uid", "filename", "projection"]]
df = reports.merge(frontal, on="uid", how="inner")
if df.empty:
    raise ValueError("No studies with both a non-empty report and frontal image were found.")
print("Usable frontal studies:", len(df))
display(df[["uid", "filename", "target_text"]].head())


In [ ]:
manifest_path = SHARED / "split_manifest.csv"
split_names = ["train", "val", "test"]
split_ratios = {"train": 0.80, "val": 0.10, "test": 0.10}
current_uids = set(df["uid"].astype(str).unique())
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path, dtype={"uid": str})
    if not {"uid", "split"}.issubset(manifest.columns):
        raise ValueError("Saved split_manifest.csv must contain 'uid' and 'split' columns.")
    manifest = manifest[["uid", "split"]].copy()
    manifest["uid"] = manifest["uid"].astype(str)
    manifest["split"] = manifest["split"].astype(str).str.lower().replace({"validation": "val"})
    if manifest["uid"].duplicated().any():
        raise ValueError("Saved split_manifest.csv contains duplicate study IDs. Fix it before continuing.")
    invalid_splits = set(manifest["split"]) - set(split_names)
    if invalid_splits:
        raise ValueError(f"Unknown split labels in manifest: {sorted(invalid_splits)}")
    extra_uids = sorted(current_uids - set(manifest["uid"]))
    if extra_uids:
        counts = manifest["split"].value_counts().reindex(split_names, fill_value=0).to_dict()
        targets = {name: split_ratios[name] * (len(manifest) + len(extra_uids)) for name in split_names}
        new_rows = []
        for uid in extra_uids:
            chosen = max(split_names, key=lambda name: (targets[name] - counts[name], -split_names.index(name)))
            new_rows.append({"uid": uid, "split": chosen})
            counts[chosen] += 1
        manifest = pd.concat([manifest, pd.DataFrame(new_rows)], ignore_index=True)
        manifest.to_csv(manifest_path, index=False)
        print(f"Added {len(extra_uids)} new study IDs; existing split assignments were preserved.")
else:
    uids = np.array(sorted(current_uids))
    np.random.default_rng(42).shuffle(uids)
    n_train, n_val = int(.8 * len(uids)), int(.9 * len(uids))
    manifest = pd.DataFrame({"uid": uids, "split": ["train"] * n_train + ["val"] * (n_val - n_train) + ["test"] * (len(uids) - n_val)})
    manifest.to_csv(manifest_path, index=False)

split_map = dict(zip(manifest.uid.astype(str), manifest.split.astype(str)))
df["uid"] = df.uid.astype(str)
df["split"] = df.uid.map(split_map)
if df.split.isna().any():
    raise ValueError("Some studies did not receive a train/val/test split.")


In [ ]:
dataset_path = SHARED / "dataset.csv"
df.to_csv(dataset_path, index=False)
for split in ("train", "val", "test"):
    df.loc[df.split.eq(split)].to_csv(SHARED / f"{split}.csv", index=False)
print("Saved shared dataset and split CSVs to", SHARED)
print(df.groupby("split").agg(studies=("uid", "nunique"), rows=("uid", "size")))
